# 섹터 강도 검증 (Colab)

**목적**: `sector_data_fetch.ipynb` 산출 parquet 을 바탕으로 ADR-003 Amendment 섹터 강도 산식을 실데이터로 검증한다.

**선행 조건**:
- `MyDrive/morning-report/sector_data/sector_map.parquet` (164행)
- `MyDrive/morning-report/sector_data/stocks_daily.parquet` (79,644행)
- 없으면 `sector_data_fetch.ipynb` 먼저 실행

**흐름**:
1. Setup (pip, drive mount, repo clone)
2. Unit tests (pytest)
3. 최신 섹터 점수 산출
4. 12개월 회귀 검증 (주도 섹터 vs KOSPI)
5. `ticker_overrides` iterate 패턴

**예상 소요**: 15-25분.

상세 runbook: `docs/plans/002-sector-breadth-pc-execution.md`

---

## 섹션 0: Setup

In [ ]:
# sector_breadth 런타임 의존성
!pip install -q pyarrow pyyaml pytest yfinance pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/morning-report/sector_data'
SECTOR_MAP = f'{DRIVE_DATA}/sector_map.parquet'
STOCKS_DAILY = f'{DRIVE_DATA}/stocks_daily.parquet'

!ls -la {DRIVE_DATA}

In [ ]:
# 레포 클론 (public 인 경우) + 작업 브랜치 checkout
# private 이면 GITHUB_TOKEN 환경변수 설정 후 https://TOKEN@github.com/... 형식 사용
import os
REPO_URL = 'https://github.com/gengar200005/morning-report.git'
BRANCH = 'claude/session-start-UZymn'
REPO_DIR = '/content/morning-report'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print('이미 클론됨 → pull')
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull

%cd {REPO_DIR}
!git log --oneline -3

## 섹션 1: 단위 테스트 (pytest)

총 30여 개 테스트. 모두 통과해야 다음 단계 진행.
실패 시: 로직 수정 → 커밋 → 이 셀부터 재실행.

In [ ]:
!pytest tests/test_sector_breadth.py -v --tb=short

## 섹션 2: 최신 시점 섹터 점수

`stocks_daily` 의 마지막 거래일 기준 15-22개 섹터 점수 출력.

**관심 포인트**:
- `주도` 등급 섹터 개수 (0-3개 예상)
- `N/A` 섹터 (비금속광물 등 표본 <3)
- `금융업` 상위권이면 지주회사 오버라이드 시급

In [ ]:
!python sector_breadth.py \
    --sector-map {SECTOR_MAP} \
    --stocks-daily {STOCKS_DAILY} \
    --overrides reports/sector_overrides.yaml

In [ ]:
# 파이썬 API 로 상세 조회 (종목 분포, 금융 편중 체감)
import sys
sys.path.insert(0, '.')
import pandas as pd
import sector_breadth as sb

sm = sb.load_sector_map(SECTOR_MAP)
sd = sb.load_stocks_daily(STOCKS_DAILY)
overrides = sb.load_overrides('reports/sector_overrides.yaml')
sm = sb.apply_ticker_overrides(sm, overrides)

print(f'오버라이드 적용: {len(overrides)} 종목')
print(f'매핑된 섹터 수: {sm["sector"].notna().sum()} / {len(sm)} 종목')
print()
print('=== 섹터별 종목 수 ===')
print(sm.groupby('sector').size().sort_values(ascending=False))

## 섹션 3: 12개월 회귀 검증

매 월말 기준 `주도` 섹터의 다음 1개월 수익률 vs KOSPI (^KS11).

**성공 기준 (ADR-003)**:
- 평균 초과수익 > 0
- hit ratio ≥ 60%

실패 시: 임계값/가중치 튜닝 or ADR-003 재검토.

In [ ]:
!python scripts/validate_sector_breadth.py \
    --sector-map {SECTOR_MAP} \
    --stocks-daily {STOCKS_DAILY} \
    --overrides reports/sector_overrides.yaml \
    --months 12 \
    --grades 주도

In [ ]:
# 참고: 주도 신호가 적으면 '주도+강세' 로도 확인
!python scripts/validate_sector_breadth.py \
    --sector-map {SECTOR_MAP} \
    --stocks-daily {STOCKS_DAILY} \
    --overrides reports/sector_overrides.yaml \
    --months 12 \
    --grades 주도,강세

## 섹션 4: ticker_overrides iterate

금융업 섹터에 지주회사가 많으면 실 사업으로 재분류.

**절차**:
1. 아래 셀로 현재 금융업 소속 종목 목록 출력
2. `reports/sector_overrides.yaml` 파일 편집 (왼쪽 파일 탭)
3. 섹션 2, 3 재실행해서 분포 변화 확인

In [ ]:
# 금융업 소속 종목 (오버라이드 적용 전·후 비교)
sm_raw = sb.load_sector_map(SECTOR_MAP)
sm_adj = sb.apply_ticker_overrides(sm_raw, sb.load_overrides('reports/sector_overrides.yaml'))

fin_raw = sm_raw[sm_raw['sector'] == '금융업'][['ticker', 'name', 'marcap']].sort_values('marcap', ascending=False)
fin_adj = sm_adj[sm_adj['sector'] == '금융업'][['ticker', 'name', 'marcap']].sort_values('marcap', ascending=False)

print(f'금융업 종목: 오버라이드 전 {len(fin_raw)} → 후 {len(fin_adj)}')
print()
print('=== 오버라이드 적용 후 금융업 Top 30 (시총순) ===')
print(fin_adj.head(30).to_string(index=False))

In [ ]:
# 현재 오버라이드 상태 확인
!cat reports/sector_overrides.yaml | tail -30

## 섹션 5: 결과 커밋 (선택)

`reports/sector_overrides.yaml` 에 반영한 오버라이드와 회귀 검증 결과를 레포에 커밋한다.
PAT 필요 (Colab secrets 또는 getpass).

In [ ]:
# 예시 (주석 처리): 필요 시 주석 해제 후 실행
#
# !git -C {REPO_DIR} config user.email 'your@email'
# !git -C {REPO_DIR} config user.name 'Your Name'
# !git -C {REPO_DIR} add reports/sector_overrides.yaml
# !git -C {REPO_DIR} commit -m 'feat(sector): ticker_overrides 실데이터 기반 확장'
#
# from getpass import getpass
# token = getpass('GitHub PAT: ')
# !git -C {REPO_DIR} push https://{token}@github.com/gengar200005/morning-report.git {BRANCH}